# Demo 02 - Identity ML anomaly detection (hero lab)

**Workshop:** F16 Advanced analytics (notebooks / ML) · **Pool:** Medium · **Time:** ~45 min

A KQL rule uses a fixed threshold ("> 100 failures"). Here we do what KQL **cannot**: build a
behavioural **feature vector per user** from history, then use an **Isolation Forest** to rank
users behaving unlike themselves and unlike their peers - the model learns the baseline.

Then we write the ranked anomalies back to a custom table for the SOC (F16 "enrichment
output").

> Set `WORKSPACE`. Increase `LOOKBACK_DAYS` for a deeper baseline (more compute).

## 1. Configuration

Everything tunable lives in one cell so you can change the shape of the run without reading
the code:

- `LOOKBACK_DAYS` - how much history defines "normal" for each user. Longer is a more
  stable baseline but slower, and slower to react to genuine change.
- `MIN_SIGNINS` - ignore users with too little activity to profile. Two sign-ins is not a
  pattern.
- `CONTAMINATION` - what fraction of users you expect to be anomalous. This is the model's
  sensitivity dial: 0.02 means "flag roughly the oddest 2%". It is a judgement call to make
  with the SOC, not a fact to discover.

In [ ]:
WORKSPACE = "your-workspace-name"   # <-- replace with your workspace name
LOOKBACK_DAYS = 30                   # baseline window; try 60/90 for depth
MIN_SIGNINS = 5                      # ignore users with too little activity to profile
CONTAMINATION = 0.02                 # expected fraction of anomalies (tune with the SOC)

## 2. Imports and connect

Spark functions, the Sentinel provider, and matplotlib.

`StructType` and `StructField` are here because two of the columns we need - `Status` and
`LocationDetails` - arrive as JSON strings rather than structured columns. We have to
describe their shape before Spark can unpack them, which happens in the next cell.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import matplotlib.pyplot as plt

data_provider = MicrosoftSentinelProvider(spark)

## 3. Ingest + normalise sign-ins

We read the interactive and (if present) non-interactive sign-in tables, restrict to the
lookback window, derive **Success/Failure** from the Entra error code, and extract the
sign-in **country** from the `LocationDetails` JSON.

In [ ]:
STATUS_SCHEMA = StructType([StructField("errorCode", StringType(), True)])
LOCATION_SCHEMA = StructType([
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("countryOrRegion", StringType(), True),
])
SUCCESS_CODES = ["0", "50125", "50140", "70043", "70044"]

def load_and_prepare(table_name):
    df = data_provider.read_table(table_name, WORKSPACE)
    # Restrict to the baseline window
    df = df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
    # Success / failure from the Status JSON error code
    df = (df.withColumn("Status_json", F.from_json(F.col("Status"), STATUS_SCHEMA))
            .withColumn("ResultCode", F.col("Status_json.errorCode"))
            .withColumn("Outcome", F.when(F.col("ResultCode").isin(SUCCESS_CODES), "Success").otherwise("Failure")))
    # Country from LocationDetails JSON
    df = (df.withColumn("Loc", F.from_json(F.col("LocationDetails"), LOCATION_SCHEMA))
            .withColumn("Country", F.col("Loc.countryOrRegion")))
    # Off-hours flag (outside 08:00-18:00 local of TimeGenerated)
    df = df.withColumn("OffHours", ((F.hour("TimeGenerated") < 8) | (F.hour("TimeGenerated") >= 18)).cast("int"))
    return df.select("UserPrincipalName", "IPAddress", "Country", "Outcome", "OffHours")

frames = [load_and_prepare("SigninLogs")]
try:
    frames.append(load_and_prepare("AADNonInteractiveUserSignInLogs"))
    print("Included AADNonInteractiveUserSignInLogs")
except Exception as e:
    print("Non-interactive table unavailable, continuing with SigninLogs only:", e)

signins = frames[0]
for extra in frames[1:]:
    signins = signins.unionByName(extra)

# Sign-ins with no UserPrincipalName (app / service-principal and some non-interactive rows)
# can't be attributed to a person. Left in, groupBy keeps them as ONE null-keyed "user"
# that skews the population, wins the anomaly ranking, and writes a junk enrichment row.
signins = signins.filter(F.col("UserPrincipalName").isNotNull() &
                         (F.trim(F.col("UserPrincipalName")) != ""))

# count() and show() are two Spark actions, and the feature build below is a third.
# Cache once so the read and filter plan is computed a single time.
signins.cache()
print("Rows in baseline window (attributable to a user):", signins.count())
signins.show(5, truncate=False)

## 4. Feature engineering - one row per user

For every user we compute the behavioural signals that, together, describe "normal":
failure ratio, breadth of source IPs and countries, off-hours share, and overall volume.

In [ ]:
features = (signins.groupBy("UserPrincipalName")
    .agg(
        F.count("*").alias("TotalSignins"),
        F.sum(F.when(F.col("Outcome") == "Failure", 1).otherwise(0)).alias("Failures"),
        F.countDistinct("IPAddress").alias("DistinctIPs"),
        F.countDistinct("Country").alias("DistinctCountries"),
        F.sum("OffHours").alias("OffHoursCount"),
    )
    .withColumn("FailureRatio", F.col("Failures") / F.col("TotalSignins"))
    .withColumn("OffHoursRatio", F.col("OffHoursCount") / F.col("TotalSignins"))
    .filter(F.col("TotalSignins") >= MIN_SIGNINS))

print("Users profiled:", features.count())
features.orderBy(F.desc("TotalSignins")).show(10, truncate=False)

## 5. Machine learning - Isolation Forest

**Isolation Forest** works on a neat idea: outliers are easy to separate from everyone else.

The model repeatedly splits the population on a randomly chosen feature at a randomly
chosen value, building a lot of small decision trees. A user with an ordinary profile sits
in the crowded middle and takes many splits to isolate. A user with a strange profile gets
cut off from everybody else after only a few. The **average number of splits needed to
isolate someone becomes their anomaly score** - fewer splits means more anomalous.

It is **unsupervised**, meaning it needs no labelled examples of attacks. That matters,
because nobody has a clean labelled dataset of compromised accounts for their own tenant.
The model learns what is normal *here* and flags what departs from it.

`contamination` tells it what fraction to flag. `random_state=42` fixes the randomness so
the same data gives the same answer twice, which you want in a demo and in production.

The per-user table is small (one row per user, not one per sign-in), so it is safe to pull
to the driver with `.toPandas()` and hand to scikit-learn. If fewer than ten users are
profiled there is no population to compare against, so the cell says so and skips.

In [ ]:
from sklearn.ensemble import IsolationForest

FEATURE_COLS = ["FailureRatio", "DistinctIPs", "DistinctCountries", "OffHoursRatio", "TotalSignins"]

# Aggregated per-user data is small - safe to bring to the driver as Pandas for sklearn.
pdf = features.toPandas()

if len(pdf) < 10:
    # Isolation Forest needs a population to compare against; don't fail the demo.
    print(f"Only {len(pdf)} users profiled - too small to model. "
          "Raise LOOKBACK_DAYS or lower MIN_SIGNINS, then re-run.")
    pdf["is_anomaly"] = False
    pdf["anomaly_score"] = 0.0
else:
    model = IsolationForest(contamination=CONTAMINATION, random_state=42)
    X = pdf[FEATURE_COLS].fillna(0.0)
    pdf["is_anomaly"] = (model.fit_predict(X) == -1)
    pdf["anomaly_score"] = -model.score_samples(X)   # higher = more anomalous

pdf = pdf.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
print(f"Flagged {int(pdf['is_anomaly'].sum())} of {len(pdf)} users as anomalous")

## 6. Review the top anomalies (and *why* they were flagged)

The score alone is not actionable - "user X scores 0.71" tells an analyst nothing. Showing
the score next to the five raw features is what makes it reviewable.

Read across each row and ask which feature is out of line with the others. A high
`DistinctCountries` with a low `FailureRatio` is a travelling executive. A high
`FailureRatio` with a high `DistinctIPs` is somebody being sprayed.

In [ ]:
cols = ["UserPrincipalName", "anomaly_score", "is_anomaly"] + FEATURE_COLS
pdf.head(20)[cols]

## 7. Visualise the top anomalous users

One bar per flagged user, tallest first.

The useful thing to look at is the *shape* of the drop-off. A few tall bars falling away
sharply means the model found a small distinct group, and your `CONTAMINATION` setting is
about right. A flat row of near-identical bars means it is slicing an arbitrary line
through a continuum, and the ranking is not telling you much.

Labels are forced to strings because matplotlib's categorical axis rejects `None`.

In [ ]:
top = pdf[pdf["is_anomaly"]].head(15)
if not top.empty:
    plt.figure(figsize=(12, 6))
    # matplotlib's categorical axis raises TypeError on None - force string labels.
    plt.bar(top["UserPrincipalName"].astype(str), top["anomaly_score"], color="#c0392b")
    plt.xlabel("User")
    plt.ylabel("Anomaly score (higher = more unusual)")
    plt.title(f"Top anomalous users - {LOOKBACK_DAYS}-day identity baseline")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No anomalies at the current contamination setting - try raising CONTAMINATION.")

## 8. Write the enrichment back for the SOC

We persist the ranked anomalies to a **lake-tier** custom table (`_SPRK`, cheapest). To let
the SOC hunt it in KQL, promote to the **analytics tier** with `_SPRK_CL` (append-only from
notebooks; needs Security Operator + the data-lake managed identity role - see setup guide).

In [ ]:
if pdf.empty:
    print("Nothing to write - no users were profiled in this window.")
else:
    result_sdf = spark.createDataFrame(pdf[cols])

    # Lake tier (cheap) - overwrite each run.
    # No database argument, so this lands in the "System tables" database: that is the
    # only lake-tier location that supports overwrite (and partitionBy). Read it back with
    #     data_provider.read_table("IdentityAnomalies_SPRK")
    run_id = data_provider.save_as_table(
        result_sdf,
        "IdentityAnomalies_SPRK",
        write_options={"mode": "overwrite"},
    )
    print("Wrote IdentityAnomalies_SPRK, run id:", run_id)

# --- Optional: promote to the analytics tier so the SOC can KQL-hunt it (append-only) ---
# Analytics tier takes the workspace as the third positional argument.
# data_provider.save_as_table(
#     result_sdf.filter("is_anomaly = true"),
#     "IdentityAnomalies_SPRK_CL",
#     WORKSPACE,
#     write_options={"mode": "append"},
# )

## Recap

- We learned each user's baseline from history and let **ML rank the outliers** - no magic
  threshold.
- Output is a ranked enrichment table the SOC can consume - the F16 "usable output".
- **Cost:** heavy read/aggregate ran in the lake tier; only the compact result is persisted.

**Stretch:** add a feature (e.g. `countDistinct("AppId")`) and watch the rankings change.
Then take this analytic into **Demo 04** to run it on a schedule.